# Entrega 1 — Análise Comparativa de Representações
**Ouvidoria Inteligente: Triagem Semântica de Manifestações Cidadãs**

Comparação entre representações esparsas (Bag-of-Words, TF-IDF) e densas (Embeddings) sobre o corpus de 40 manifestações da ouvidoria, incluindo o cálculo de similaridade de cosseno para os pares indicados no enunciado.

## 1. Carregamento do corpus

In [1]:
import json
import numpy as np
import pandas as pd

with open("manifestacoes.json", encoding="utf-8") as f:
    manifestacoes = json.load(f)

df = pd.DataFrame(manifestacoes)
textos = df["texto"].tolist()
ids = df["id"].tolist()
id_to_idx = {i: idx for idx, i in enumerate(ids)}

print(f"{len(df)} manifestações carregadas.")
df.head()

40 manifestações carregadas.


,id,data,categoria_oficial,texto
0,M001,2026-01-12,infraestrutura,A rua Sete de Setembro está sem manutenção há ...
1,M002,2026-01-15,meio ambiente,Moradores estão despejando lixo doméstico às m...
2,M003,2026-01-18,infraestrutura,"Existe um buraco enorme na Av. Brasil, próximo..."
3,M004,2026-01-20,educação,A Escola Municipal José de Alencar está sem pr...
4,M005,2026-01-22,saúde,Sou moradora do bairro Vila Nova e gostaria de...


## 2. Representação Esparsa: Bag-of-Words (BoW)
BoW conta a frequência bruta de cada termo, sem qualquer noção de importância relativa ou de significado.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow = CountVectorizer()
X_bow = bow.fit_transform(textos)
print(f"Vocabulário BoW: {len(bow.vocabulary_)} termos")
print(f"Matriz BoW: {X_bow.shape} (manifestações × termos)")

Vocabulário BoW: 659 termos
Matriz BoW: (40, 659) (manifestações × termos)


## 3. Representação Esparsa: TF-IDF
TF-IDF pondera os termos pela sua raridade na coleção, reduzindo o peso de palavras muito comuns.

In [3]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(textos)
print(f"Matriz TF-IDF: {X_tfidf.shape}")

Matriz TF-IDF: (40, 659)


## 4. Representação Densa: Embeddings
Usamos um modelo multilíngue de *sentence embeddings*, capaz de capturar o significado da frase (não apenas as palavras que ela contém).

In [4]:
from sentence_transformers import SentenceTransformer

NOME_MODELO = "paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(NOME_MODELO)
X_emb = modelo.encode(textos, show_progress_bar=True, normalize_embeddings=True)
print(f"Matriz de Embeddings ({NOME_MODELO}): {X_emb.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Matriz de Embeddings (paraphrase-multilingual-MiniLM-L12-v2): (40, 384)


## 5. Comparação de Similaridade de Cosseno — Pares do Enunciado
- **M003 × M017**: "buraco enorme na Av. Brasil" × "asfalto todo esburacado da avenida principal" — mesma reclamação, vocabulário diferente.
- **M008 × M022**: "posto de saúde sem médico" × "falta atendimento no PSF" — mesmo problema (falta de médico), termos técnicos diferentes (PSF).
- **M008 × M031**: "posto de saúde sem médico" × "lâmpada queimada na praça" — problemas completamente distintos (par de controle).

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def cosseno(matriz, i, j):
    """Similaridade de cosseno entre as linhas i e j de uma matriz esparsa ou densa."""
    vi = matriz[i]
    vj = matriz[j]
    if hasattr(vi, "toarray"):
        vi, vj = vi.toarray(), vj.toarray()
    else:
        vi, vj = vi.reshape(1, -1), vj.reshape(1, -1)
    return float(cosine_similarity(vi, vj)[0][0])

pares = [("M003", "M017"), ("M008", "M022"), ("M008", "M031")]

linhas = []
for a, b in pares:
    ia, ib = id_to_idx[a], id_to_idx[b]
    linhas.append({
        "Par": f"{a} × {b}",
        "BoW": round(cosseno(X_bow, ia, ib), 4),
        "TF-IDF": round(cosseno(X_tfidf, ia, ib), 4),
        "Embeddings": round(cosseno(X_emb, ia, ib), 4),
    })

df_comparacao = pd.DataFrame(linhas)
df_comparacao

,Par,BoW,TF-IDF,Embeddings
0,M003 × M017,0.1886,0.1232,0.5373
1,M008 × M022,0.2619,0.1713,0.7005
2,M008 × M031,0.2294,0.1004,0.2827


### Textos completos dos pares (para referência na discussão)

In [6]:
for a, b in pares:
    print(f"--- {a} × {b} ---")
    print(f"{a}: {df.loc[id_to_idx[a], 'texto']}")
    print(f"{b}: {df.loc[id_to_idx[b], 'texto']}")
    print()

--- M003 × M017 ---
M003: Existe um buraco enorme na Av. Brasil, próximo ao número 450, que já causou danos a pneus de vários veículos que passam pelo local.
M017: O asfalto está todo esburacado na avenida principal do centro, dificultando o tráfego e causando vários acidentes de moto no local.

--- M008 × M022 ---
M008: O posto de saúde do bairro Alto do Mateus está sem médico há semanas, deixando a população sem atendimento clínico básico na região.
M022: Está faltando atendimento no PSF do meu bairro, os pacientes chegam cedo e não conseguem ser vistos por nenhum médico durante o dia.

--- M008 × M031 ---
M008: O posto de saúde do bairro Alto do Mateus está sem médico há semanas, deixando a população sem atendimento clínico básico na região.
M031: A lâmpada do poste em frente à praça central está queimada há semanas e ninguém da prefeitura veio fazer a substituição até agora.



## 6. Discussão dos Resultados

**Resultados reais obtidos (modelo `paraphrase-multilingual-MiniLM-L12-v2`, executado com conexão à internet):**

| Par | BoW | TF-IDF | Embeddings |
|---|---|---|---|
| M003 × M017 | 0.1886 | 0.1232 | 0.5373 |
| M008 × M022 | 0.2619 | 0.1713 | 0.7005 |
| M008 × M031 | 0.2294 | 0.1004 | 0.2827 |

**M008 × M022** (mesmo problema, com um termo técnico do domínio — "PSF"): o padrão esperado se confirma com folga. A sobreposição lexical é baixa (BoW e TF-IDF ≈ 0.17–0.26), mas os Embeddings sobem para 0.70, reconhecendo que "posto de saúde sem médico" e "falta atendimento no PSF" descrevem o mesmo problema mesmo com vocabulário técnico diferente.

**M008 × M031** (par de controle, temas não relacionados): o padrão esperado também se confirma — os três métodos convergem para similaridade baixa (0.10–0.28), já que não há sobreposição lexical nem semântica relevante entre "falta de médico" e "lâmpada queimada".

**M003 × M017** (mesmo problema, vocabulário diferente — "buraco" vs. "esburacado"/"avenida principal"): o padrão se confirma apenas **parcialmente**. Os Embeddings (0.5373) ficam bem acima de BoW e TF-IDF, mostrando que o modelo capta alguma relação semântica entre os dois textos — mas 0.54 está na faixa de similaridade **moderada**, não alta, pela mesma escala usada no restante do relatório e no app (moderado: 0.5–0.7; alto: acima de 0.7). Uma explicação plausível está no próprio texto das manifestações: M003 cita um local específico ("Av. Brasil, nº 450") e M017 fala de forma mais genérica ("avenida principal do centro") — o modelo não tem como saber se são o mesmo trecho de rua ou não, e por isso hesita entre tratá-los como a mesma reclamação ou como reclamações parecidas, mas distintas. Isso reforça um limite real dos embeddings: eles captam semelhança de significado, não identidade factual (mesmo endereço, mesmo órgão, mesma data) — um ponto retomado na Entrega 2, ao discutir o limiar de detecção de duplicatas.

### Limitações de cada abordagem

- **Bag-of-Words**: ignora completamente a ordem das palavras e não possui noção de sinônimos ou de importância relativa dos termos — duas frases sobre o mesmo assunto, mas escritas com vocabulário diferente, são tratadas como quase ortogonais no espaço vetorial.
- **TF-IDF**: resolve parcialmente o problema de termos muito frequentes (como "o", "de", "que"), mas continua sendo uma representação esparsa e lexical — ainda não reconhece que "buraco" e "esburacado" ou "médico" e "atendimento" se referem ao mesmo problema.
- **Embeddings**: capturam relações semânticas e sinônimos, sendo a abordagem que melhor identifica duplicatas reais na ouvidoria — mas, como o par M003×M017 mostra, isso não é garantido: quando os textos mencionam detalhes concretos diferentes (endereços, por exemplo), a similaridade pode ficar apenas moderada mesmo tratando-se do mesmo problema. É também a abordagem computacionalmente mais custosa, que exige um modelo pré-treinado (dependência externa) e pode, em casos raros, aproximar textos que soam parecidos mas tratam de assuntos distintos (falsos positivos), algo que TF-IDF dificilmente faria para termos muito específicos (como códigos ou nomes próprios).